# 🎨 Open Image Edit Studio: SDXL on your own Colab

Edit an image with a text prompt using **SDXL**, running on **your own free Colab GPU**. No Hugging Face quota is used.

**Steps**
1. **Runtime → Change runtime type → T4 GPU** → Save.
2. Run each cell in order (▶). Cell 2 downloads the model the first time, which takes 3–5 minutes.
3. Upload your image in cell 3, type your prompt in cell 4, run it, and the result downloads automatically.

Change the prompt or settings in cell 4 and run it again as many times as you like; you don't need to reload the model.

| Mode | Model | License | Best for |
|---|---|---|---|
| **Instruction edit** | `diffusers/sdxl-instructpix2pix-768` | OpenRAIL++ | "make it night", "turn the car red" |
| **Img2img restyle** | `stabilityai/stable-diffusion-xl-base-1.0` | OpenRAIL++ | describe the *target* image; restyle the whole picture |

Web app: https://bibek-cse.github.io/image-edit-studio/

In [ ]:
#@title 1️⃣ Check GPU & install libraries
import subprocess, sys
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("❌ No GPU found. Go to Runtime → Change runtime type → select T4 GPU, then run this cell again.")
print("✅ GPU:", gpu)
!pip -q install -U diffusers transformers accelerate safetensors
print("✅ Libraries ready")

In [ ]:
#@title 2️⃣ Load model
mode = "Instruction edit (SDXL InstructPix2Pix)" #@param ["Instruction edit (SDXL InstructPix2Pix)", "Img2img restyle (SDXL base 1.0)"]

import torch, gc
from diffusers import (StableDiffusionXLInstructPix2PixPipeline,
                       StableDiffusionXLImg2ImgPipeline, AutoencoderKL)

if "pipe" in globals():
    del pipe; gc.collect(); torch.cuda.empty_cache()

# fp16-safe SDXL VAE (prevents black / NaN images in half precision)
vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)

if mode.startswith("Instruction"):
    MODEL_ID, RES = "diffusers/sdxl-instructpix2pix-768", 768
    pipe = StableDiffusionXLInstructPix2PixPipeline.from_pretrained(
        MODEL_ID, vae=vae, torch_dtype=torch.float16, use_safetensors=True)
else:
    MODEL_ID, RES = "stabilityai/stable-diffusion-xl-base-1.0", 1024
    pipe = StableDiffusionXLImg2ImgPipeline.from_pretrained(
        MODEL_ID, vae=vae, torch_dtype=torch.float16, variant="fp16", use_safetensors=True)

pipe.to("cuda")
pipe.enable_vae_slicing()
pipe.set_progress_bar_config(desc="Generating")
LOADED_MODE = mode
print(f"✅ Loaded {MODEL_ID}  (working size ≈ {RES}px)")

In [ ]:
#@title 3️⃣ Upload your image
from google.colab import files
from PIL import Image, ImageOps
from IPython.display import display
import io

uploaded = files.upload()
if not uploaded:
    raise SystemExit("No file uploaded.")
name, data = next(iter(uploaded.items()))
src = ImageOps.exif_transpose(Image.open(io.BytesIO(data))).convert("RGB")

def fit(img, target):
    """Scale so the longer side = target, both sides multiples of 8."""
    s = target / max(img.size)
    w, h = (max(8, int(round(d * s / 8)) * 8) for d in img.size)
    return img.resize((w, h), Image.LANCZOS)

input_image = fit(src, RES)
print(f"✅ {name}: original {src.size} → model input {input_image.size}")
preview = input_image.copy(); preview.thumbnail((384, 384)); display(preview)

In [ ]:
#@title 4️⃣ Prompt & generate (re-run freely)
prompt = "make it a snowy night" #@param {type:"string"}
negative_prompt = "blurry, low quality, distorted" #@param {type:"string"}
steps = 30 #@param {type:"slider", min:10, max:60, step:1}
guidance_scale = 5.0 #@param {type:"slider", min:1, max:12, step:0.5}
#@markdown **Instruction edit only:** how closely to keep the original image (higher = fewer changes)
image_guidance_scale = 1.5 #@param {type:"slider", min:1, max:3, step:0.1}
#@markdown **Img2img only:** how much to change the image (higher = more change)
strength = 0.6 #@param {type:"slider", min:0.1, max:1.0, step:0.05}
seed = -1 #@param {type:"integer"}
num_images = 1 #@param {type:"slider", min:1, max:4, step:1}
auto_download = True #@param {type:"boolean"}

import os, random, time
assert "input_image" in globals(), "Run cell 3 first to upload an image."
if seed < 0:
    seed = random.randint(0, 2**31 - 1)
g = torch.Generator("cuda").manual_seed(seed)

common = dict(prompt=prompt, negative_prompt=negative_prompt or None, image=input_image,
              num_inference_steps=steps, guidance_scale=guidance_scale,
              num_images_per_prompt=num_images, generator=g)
t0 = time.time()
if LOADED_MODE.startswith("Instruction"):
    out = pipe(**common, image_guidance_scale=image_guidance_scale).images
else:
    out = pipe(**common, strength=strength).images

os.makedirs("/content/outputs", exist_ok=True)
stamp = time.strftime("%Y%m%d-%H%M%S")
paths = []
for i, im in enumerate(out):
    p = f"/content/outputs/sdxl_{stamp}_seed{seed}_{i+1}.png"
    im.save(p); paths.append(p)
print(f"✅ Done in {time.time()-t0:.1f}s · seed {seed}")
for p in paths:
    show = Image.open(p).copy(); show.thumbnail((640, 640)); display(show)
if auto_download:
    for p in paths:
        files.download(p)

In [ ]:
#@title 5️⃣ (Optional) Download all results as a ZIP
import shutil
zip_path = shutil.make_archive("/content/sdxl_results", "zip", "/content/outputs")
files.download(zip_path)

**Notes**
* Colab's free GPU is for interactive use. Run the cells yourself, as above.
* Image too large or out of memory? Restart the runtime and use a smaller image.
* Each model keeps its own license (both here are OpenRAIL++). Don't use edits to deceive or impersonate real people.